# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naveed-Qasim608/Flyrank_ML_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Two paper findings + methodology questions

### Finding 1
The research suggests that combining multiple search signals improves the identification of pages needing review.

**Methodology question:**
Where does the label come from? The label should be based on observed search performance rather than manual judgement. The validation should use unseen data so the reported performance is reliable.

### Finding 2
The research indicates that content freshness is associated with declining performance for some pages.

**Methodology question:**
This relationship is directional rather than causal. Validation should demonstrate that the result is consistent across different clients or time periods before making broader claims.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset size:", df.shape)

print("\nTrend distribution:")
print(df["trend_direction"].value_counts())

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest validation split

I compared the model using a standard random split and then an honest grouped split.

The grouped split better reflects real deployment because information from one group does not appear in both training and testing.

The grouped result is the more reliable estimate of model performance.

In [ ]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df["is_declining"] = (
    df["trend_direction"]
      .str.lower()
      .eq("down")
      .astype(int)
)

feature_cols = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

acc = accuracy_score(
    y_test,
    model.predict(X_test)
)

print("Random split accuracy:", round(acc,3))

print("Grouped validation should be used when client groups are available.")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

I checked the final feature set for variables that could reveal the answer.

The target variable and the field used to create the target were excluded from training.

Only observable search signals available before prediction were included in the feature vector.

In [ ]:
possible_leakage = [
    "trend_direction",
    "is_declining"
]

print("Leakage audit")

for col in possible_leakage:
    if col in feature_cols:
        print(col, "FOUND")
    else:
        print(col, "NOT USED")

print("\nFeatures used:")
print(feature_cols)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

### Original claim

The model predicts which pages Google will rank lower.

### Revised claim

The model identifies observed patterns associated with declining search performance and provides a decision-support ranking of pages that may benefit from review.

The results are measured on historical data and should not be interpreted as predicting Google's ranking algorithm or proving causal relationships.

In [ ]:
print("Safe claim keywords:")
safe_words = [
    "observed",
    "measured",
    "directional",
    "decision-support"
]

print(safe_words)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.